In the following discussion, we will equate a Pokemon with its type, stats, and four known moves.  There's more to it, of course, but this seems like a good place to start.

# To Dos
- Solve the forme issue regarding stats

# The Problem

A pokemon's type, stats, and four known moves gives 12 dimensions of information about that pokemon.  While Player 1 win probability should be a nondecreasing function of the numeric stats, the amount of benefit provided from increasing a stat by one point depends on the matchup.  If you replace Weavile by Chien-Pao, your win probability should go up!  But how much it goes up should depend on the matchup.  If your opponent has no pokemon between Weavile's base 125 speed and Chien-Pao's base 135 speed, then your win probability shouldn't go up by very much.  Or, if your oppenent has lots of pokemon with high physical defense, then Chien-Pao's power increase (coming from its ability, not its stats) may not make much of a difference either.  But if your opponent has Eternatus (base speed 130), then replacing Weavile by Chien-Pao should make a big difference!  So a statistic's contribution to win probability is matchup dependent, and this seems hard for a model to learn.

Another issue is that for nearly every pokemon (maybe we should quantify this claim), their secondary attacking stat is irrelevant.  It does not matter what your Chien-Pao's special attack stat is; it will not affect the battle at all.  So when we're training a model, the model should not be told Chien-Pao's special attack stat.  One possibility is to replace every pokemon's attack and special attack stat with offensive_stat = max(atk, spa) and record a boolean offensive_stat_is_phys.  This is okay, but it seems like the model might have a hard time learning that if offensive_stat_is_phys == True, then offensive_stat should be compared with its opponents def stats, not spd stats.

Yet another issue is that the model will have to somehow divine the type chart from these battle outcomes.  This isn't a situation where having more Psychic types is better than having fewer; once again, it's matchup-dependent.

# A Solution

Rather than giving the model a bunch of basic information (types, stats, known moves) and hoping that the model learns the stuff we already know (Fire > Grass > Water > Fire), let's tell the model the stuff that we already know.  But how do you feed in the type chart and whatnot into the model?  Seems hard.

Instead, let's bake all of our knowledge into a set of offensive and defensive stats (we'll call these "advantage" stats) that are derived for each specific battle.  Then, we'll replace the basic stats with these "advantage" stats.

We will start off by neglecting the movepool.  We could try to map each move to its category, type, and base power, but that could be a lot of work.  It's something to explore in the future.

## Damage Approximator

The advantage stat starts with an approximation of damage.  Given a pokemon mon1 on team 1 and a pokemon mon2 on team 2, we can estimate the average damage that mon1 would do to mon2 by selecting its best STAB move (or a not-very-effective coverage move in the event that is better--this could be updated if we get data suggesting that in the event that a mon's best move is a coverage move, the coverage move is often neutral or better).  Note that we assume a base power of 80 for all moves.

We approximate that when mon1 attacks mon2, mon2 will lose the following fraction of its HP:

$$\text{damage}(\text{mon1},\text{mon2}) := \left(\frac{\left(\frac{2 \cdot \text{mon1}_{\text{level}}}{5} + 2\right) \cdot 80 \cdot \frac{\text{mon1}_{\text{off-stat}}}{\text{mon2}_{\text{def-stat}}}}{50} + 2\right) \cdot \text{type-multiplier}(\text{mon1},\text{mon2}) \cdot 92.5/100 / \text{mon2}_{\text{hp}}$$

where 

$$\text{mon1}_{\text{off-stat}} = \max(\text{mon1}_{\text{atk}}, \text{mon1}_{\text{spa}}),$$

$$ \text{mon2}_{\text{def-stat}} = \begin{cases} \text{mon2}_{\text{def}} & \text{if } \text{mon1}_{\text{off-stat}} = \text{mon1}_{\text{atk}} \\ \text{mon2}_{\text{spd}} & \text{else} \end{cases}, $$

$$ \text{type-multiplier}(\text{mon1},\text{mon2}) = \max\left(\frac{1}{2}, \max_{\text{type1} \in \text{mon1}_{\text{types}}} 1.5 \cdot \prod_{\text{type2} \in \text{mon2}_{\text{types}}} \text{eff}(\text{type1},\text{type2})\right) $$

with $\text{eff}(\text{type1},\text{type2})$ being determined by the type chart, so that, e.g. $\text{eff}(\text{water},\text{fire}) = 2$ while $\text{eff}(\text{fire},\text{water}) = 1/2$.

Some notes:
- This fraction is allowed to exceed 1 as the amount by which it exceeds 1 may actually matter (think: reflect/light screen/aurora veil or resistance berries).
- This uses a simplified version of the damage formula found here: https://bulbapedia.bulbagarden.net/wiki/Damage#Generation_V_onward
- The number 80 corresponds to the base power of the move theoretically being selected.
- The use of 92.5/100 corresponds to the average value of the 'random' factor.
- The use of type-multiplier is meant to approximate the product of STAB and Type.
- type-multiplier assumes that mon1 is only using STABs.  This could be updated to account for coverage moves in a future iteration on this stat.
- The max(1/2, stuff) in type-multiplier is to prevent type-multiplier from ever being 0.  It is very rare (though it does happen) that mon1 will be unable to damage mon2.  The factor of 1/2 is used because that is a multiplier for a not-very-effective coverage move.  This would be resolved by replacing the max over mon1 types by a max over mon1 move types.
- Damage or speed boosting items will not be accounted for.  This could be resolved in an ad-hoc way by checking for common boosting items (choice items, life orb).  It could also be resolved in a systemic way be using the smogon damage calculator to replace the offensive advantage stat.
- Damage/stat modifying abilities like Levitate, Thick Fat, or Sword of Ruin will not be accounted for.  This could 'only' be resolved by using the smogon damage calculator to replace this offensive advantage stat.
- Type chart modifying moves like Freeze-Dry are not accounted for.


## Advantage

The only (relevant) thing that doesn't go into the damage approximator is speed.  Speed is difficult to incorporate into advantage.  There are a few reasons for this:
  1. The only important feature of speed differential (meaning $\text{mon1}_{\text{spe}} - \text{mon2}_{\text{spe}}$) is its sign.  Magnitude is meaningless here.  So multiplying the damage approximation by speed differential would be a bad idea.
  2. The impact of speed differential can be large or small.  If you consider a hypothetical Weavile versus Iron Boulder matchup, each has a super-effective STAB on the other (meaning it has a type-multiplier equal to 3)!  In that situation, Weavile has the advantage because it goes first.  However, if you consider a Weavile versus Swampert matchup (where each has a type-multiplier of 1.5), the Swampert has the advantage in spite of its speed disadvantage due to its overall bulk.  My initial thought is that speed matters a lot when both pokemon are doing about the same amount of damage to one another, but doesn't matter very much when the pokemon are doing very different amounts of damage.  So 'having a speed advantage' should not correspond to a constant factor.

Also worth noting is that advantage depends not just on how much damage you're doing to your opponent, but how much damage your opponent is doing to you!

Maybe try computing 'turns to KO' for each mon and look at differential.  So we get something like

$$ \text{time-to-ko-diff}(\text{mon1},\text{mon2}) = \lceil \frac{1}{\min(1,\text{damage-approx}(\text{mon2},\text{mon1}))}\rceil - \lceil\frac{1}{\min(1,\text{damage-approx}(\text{mon1},\text{mon2}))}\rceil $$

Here, bigger is better for mon1.

Properties that I want:
- There should be a nice relationship between adv(mon1,mon2) and adv(mon2,mon1) (a + b = 1 with 0 < a,b < 1?  ab = 1?)
- If time-to-ko-diff is close to 0 and the times to ko are close to 1, the faster mon should have a big advantage (this is the situation where the faster mon just OHKOs the slower mon with no cost)
- If the time-to-ko-diff is close to 0 and the times to ko are large, the faster mon should have a small advantage (this is the situation where the faster mon KOs the slower mon but at an HP cost)
- If the time-to-ko-diff is large, the mon with the smaller time to ko should have a big advantage (this is the situation where one mon just dominates the other)

So maybe the advantage stat should represent something like: expected damage dealt to opponent in a 1v1 matchup?  If we let $n$ denote the round number in which the KO occurs, then the faster mon gets to go $n$ times and the slower mon gets to go $n$ or $n-1$ times depending on who wins.

Let's set

$$\text{time-to-ko}(\text{mon1},\text{mon2}) = \lceil \frac{1}{\min(1,\text{damage-approx}(\text{mon1},\text{mon2}))} \rceil$$

Then

$$\text{turn-of-ko}(\text{mon1},\text{mon2}) = \min\left(\text{time-to-ko}(\text{mon1},\text{mon2}), \text{time-to-ko}(\text{mon2},\text{mon1})\right)$$

So

$$
\text{1v1-damage}(\text{mon1},\text{mon2}) =
\begin{cases}
    (\text{turn-of-ko}(\text{mon1},\text{mon2}) - 1) \cdot \text{damage-approx}(\text{mon1},\text{mon2})  &\text{if } \text{mon1}_{\text{spe}} < \text{mon2}_{\text{spe}} \text{ and mon2 KOs mon1}\\
    \text{turn-of-ko}(\text{mon1},\text{mon2}) \cdot \text{damage-approx}(\text{mon1},\text{mon2})        &\text{else}
    
\end{cases}
$$

Then we can do something like set $\text{adv}(\text{mon1},\text{mon2}) = \text{1v1-damage}(\text{mon1},\text{mon2})$ or we can do something fancy and make it symmetric like $\text{adv}(\text{mon1},\text{mon2}) = \text{1v1-damage}(\text{mon1},\text{mon2}) - \text{1v1-damage}(\text{mon2},\text{mon1})$.

Regardless, this should be enough to get started.


## Potential problems with advantage stats

Some pokemon are not good because of their stats.  Take Sableye for example.  It has atrocious stats, but can win a match on the strength of its ability, Prankster.  These advantage stats won't account for that.  (On the other hand, neither will training on 12-dimensional info above.)

Other pokemon don't rely on their offensive stats for damage (think Toxapex).

Yet more pokemon rely heavily on priority moves.

# Testing

In order to test the FullPokemon class, we need:

- type_multiplier:
  - m1 has a 4x effective STAB (m1 = Weavile, m2 = Salamence)
  - m1 has a 2x effective STAB (m1 = Weavile, m2 = Haxorus)
  - m1's best STAB is neutral (m1 = Weavile, m2 = Corviknight)
  - m1's best STAB is 1/2x effective (m1 = Weavile, m2 = Chien-Pao)
  - m1's best STAB is 1/4x effective (m1 = Conkeldurr, m2 = Fezandipiti)
  - m1's best STAB is 0x effective (m1 = Banette, m2 = Wigglytuff)

- damage
  - already tested manually by comparing physical and special attackers on the calcs at calc.pokemonshowdown.com

- one_v_one_damage
  - m1 is faster than m2 and KOs m2 (m1 = Weavile, m2 = Salamence)
  - m2 is faster than m1 and KOs m1 (m1 = Sinistcha?, m2 = Weavile)
  - m1 is faster than m2 yet is KOd by m2 (m1 = Weavile, m2 = Conkeldurr)
  - m2 is faster than m1 yet is KOd by m1 (m1 = Swampert, m2 = Corviknight?)
  - m1 and m2 have a speed tie (m1 = Lanturn, m2 = Toxtricity)

In [1]:
import sys
import os
import json
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import copy
from pathlib import Path
from sklearn.model_selection import StratifiedKFold,cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss,accuracy_score,roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
sys.path.insert(0,os.path.abspath('..'))
from tools import battle as b

log_dir = Path("../data/replays/gen9-randombattle")
stat_names = ['hp','atk','def','spa','spd','spe']

In [11]:
desired_mon_names = ['Weavile','Salamence','Haxorus','Corviknight','Chien-Pao','Conkeldurr','Fezandipiti','Banette','Wigglytuff','Sinistcha','Swampert','Lanturn','Toxtricity','Ditto','Palafin','Terapagos','Regigigas']
desired_item_names = ["Light Ball","Choice Band","Choice Specs","Choice Scarf","Eviolite","Assault Vest","Soul Dew"]
log_dir = Path("../data/replays/gen9-randombattle")
desired_team_dict = {}
for file in log_dir.iterdir():
    if file.is_file():
        with open(file,"r") as battle_json:
            battle = json.load(battle_json)
        for team_index in range(2):
            for mon in battle["teams_full"][team_index].keys():
                mon_is_desired = (mon in desired_mon_names)
                item_is_desired = (battle["teams_full"][team_index][mon]["item"] in desired_item_names)
                if mon_is_desired or item_is_desired:
                    desired_team_dict[mon] = battle["teams_full"][team_index][mon]
                    if mon_is_desired:
                        desired_mon_names.remove(mon)
                    if item_is_desired:
                        desired_item_names.remove(battle["teams_full"][team_index][mon]["item"])


In [18]:
desired_item_names

[]

In [19]:
desired_mon_names

[]

In [13]:
test_pokemon = {mon:b.FullPokemon(desired_team_dict[mon]) for mon in desired_team_dict.keys()}

In [17]:
test_pokemon["Toxtricity"].item

'Choice Scarf'

In [ ]:
print("Testing Individual Pokemon")
assert(test_pokemon["Palafin"].stats["atk"] == 291), "Palafin failed"
assert(test_pokemon["Terapagos"].stats["spd"] == 214), "Terapagos failed"
assert(test_pokemon["Regigigas"].stat_multiplier("atk") == 0.5), "Regigigas atk failed"
assert(test_pokemon["Regigigas"].stat_multiplier("spa") == 1), "Regigigas spa failed"
assert(test_pokemon["Pikachu"].stat_multiplier("spa") == 2), "Pikachu spa failed"
assert(test_pokemon["Pikachu"].stat_multiplier("def") == 1), "Pikachu def failed"

print("Testing items")
assert(test_pokemon["Toxtricity"].stat_multiplier("spa") == 1), "Toxtricity (Choice Scarf) spa failed"
assert(test_pokemon["Toxtricity"].stat_multiplier("spe") == 1.5), "Toxtricity (Choice Scarf) spe failed"
assert(test_pokemon["Banette"].damage_multiplier() == 5324/4096), "Banette (Life Orb) failed"

print("Now testing type_multiplier")
assert(b.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Salamence']) == 6),"Weavile and Salamence failed"
assert(b.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Haxorus']) == 3), "Weavile and Haxorus failed"
assert(b.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Corviknight']) == 1.5), "Weavile and Corviknight failed"
assert(b.FullPokemon.type_multiplier(test_pokemon['Weavile'],test_pokemon['Chien-Pao']) == 0.75), "Weavile and Chien-Pao failed"
assert(b.FullPokemon.type_multiplier(test_pokemon['Conkeldurr'],test_pokemon['Fezandipiti']) == 1/2), "Conkeldurr and Fezandipiti failed"
assert(b.FullPokemon.type_multiplier(test_pokemon['Banette'],test_pokemon['Wigglytuff']) == 1/2), "Banette and Wigglytuff failed"

print()
print("Now testing one_v_one_damage")
ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Salamence'])
assert(ovo[0] >= 1 and ovo[1] == 0), "Weavile and Salamence failed"
ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Sinistcha'],test_pokemon['Weavile'])
assert(ovo[0] > 0 and ovo[0] < 1 and ovo[1] >= 1), "Sinistcha and Weavile failed"
ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Weavile'],test_pokemon['Conkeldurr'])
assert(ovo[0] > 0 and ovo[0] < 1 and ovo[1] >=1),"Weavile and Conkeldurr failed"
ovo = b.FullPokemon.one_v_one_damage(test_pokemon['Swampert'],test_pokemon['Corviknight'])
assert(ovo[0] >= 1 and ovo[1] > 0 and ovo[1] < 1), "Swampert and Corviknight failed"
ovo = b.FullPokemon.one_v_one_damage(test_pokemon["Lanturn"],test_pokemon["Toxtricity"])
assert(ovo[0] >= 1 and ovo[1] >= 1), "Lanturn and Toxtricity failed"

Testing Individual Pokemon
Testing items


AssertionError: Toxtricity (Choice Specs) spa failed

## Examples

In [5]:
id = "2631906096"
with open("../data/replays/gen9-randombattle/gen9randombattle-" + id + ".json","r") as battle_json:
    data = json.load(battle_json)

team1 = [b.FullPokemon(data["teams_full"][0][key]) for key in data["teams_full"][0].keys()]
team2 = [b.FullPokemon(data["teams_full"][1][key]) for key in data["teams_full"][1].keys()]
col_names = [f"adv_over_{mon2.name}" for mon2 in team2]
col_names.insert(0,"p1_mon")
rows = []
for i,mon1 in enumerate(team1):
    rows.append([b.FullPokemon.advantage(mon1,mon2) for mon2 in team2])
    rows[i].insert(0,mon1.name)
df = pd.DataFrame(rows,columns=col_names)
adv_matrix = [[b.FullPokemon.advantage(mon1,mon2) for mon2 in team2] for mon1 in team1]
sum(sum(adv_matrix[i]) for i in range(len(adv_matrix)))

4.815082462271961

In [6]:
df

,p1_mon,adv_over_Abomasnow,adv_over_Ceruledge,adv_over_Chansey,adv_over_Hippowdon,adv_over_Carbink,adv_over_Klefki
0,Quaquaval,0.715086,1.328708,1.100367,0.649726,-0.168379,-0.605830
1,Pecharunt,1.050588,0.624728,0.492110,-0.674954,-0.312160,0.255685
2,Noivern,-0.756319,0.257875,-0.347247,0.900206,-0.794214,-0.862681
3,Azumarill,-1.031641,-0.066877,-0.567092,0.680872,0.350973,-0.343552
4,Gogoat,-0.768560,-1.037238,1.039033,0.872012,0.853608,-0.258471
5,Klawf,1.121944,1.654554,1.111303,-0.789109,0.170756,-1.030727


In [7]:
[[f"{adv_matrix[i][j]:.2f}" for j in range(6)] for i in range(6)]

[['0.72', '1.33', '1.10', '0.65', '-0.17', '-0.61'],
 ['1.05', '0.62', '0.49', '-0.67', '-0.31', '0.26'],
 ['-0.76', '0.26', '-0.35', '0.90', '-0.79', '-0.86'],
 ['-1.03', '-0.07', '-0.57', '0.68', '0.35', '-0.34'],
 ['-0.77', '-1.04', '1.04', '0.87', '0.85', '-0.26'],
 ['1.12', '1.65', '1.11', '-0.79', '0.17', '-1.03']]

## EDA

In [8]:
rows = []
for file in log_dir.iterdir():
    if file.is_file():
        battle = b.Battle(file)
        if not battle.custom_ruleQ: # throw out battles with custom rules
            # with open(file,"r") as battle_json:
            #     battle_data = json.load(battle_json)
            # basic data
            to_add = {
                "id": battle.id,
                "rated": battle.rated,
                "duration": battle.end_time - battle.start_time,
                "p1": battle.p1.name,
                "p2": battle.p2.name,
                "p1_rating" : battle.p1.elo0,
                "elo_diff": battle.p1.elo0 - battle.p2.elo0,
                "p1_wins" : battle.p1.name == battle.winner.name,
                "p1_revealed_team_size" : len(battle.teams[0].keys()),
                "p2_revealed_team_size" : len(battle.teams[1].keys())
            }
            # Team construction
            team1 = [b.FullPokemon(battle.teams_full[0][mon]) for mon in battle.teams_full[0].keys()]
            team2 = [b.FullPokemon(battle.teams_full[1][mon]) for mon in battle.teams_full[1].keys()]
            teams = [team1,team2]

            # stats for each mon
            for player_index in range(2):
                for mon_index in range(6):
                    to_add[f"p{player_index + 1}_m{mon_index + 1}_species_id"] = teams[player_index][mon_index].speciesId
                    to_add[f"p{player_index + 1}_m{mon_index + 1}_type_1"] = teams[player_index][mon_index].types[0]
                    try:
                        to_add[f"p{player_index + 1}_m{mon_index + 1}_type_2"] = teams[player_index][mon_index].types[1]
                    except IndexError:
                        pass
                    for stat in stat_names:
                        to_add[f"p{player_index + 1}_m{mon_index + 1}_{stat}"] = teams[player_index][mon_index].stats[stat]

            # advantage stats for each pair
            for p1_mon_index in range(6):
                for p2_mon_index in range(6):
                    to_add[f"p1_m{p1_mon_index + 1}_adv_over_p2_m{p2_mon_index+1}"] = b.FullPokemon.advantage(team1[p1_mon_index],team2[p2_mon_index])

            rows.append(to_add)

# contains all information about all relevant matches in our data set
full_match_data = pd.DataFrame(rows)

In [9]:
# The following lines are for grabbing subsets of features easily
type_col_names = [f"p{i+1}_m{m+1}_type_{t+1}" for i in range(2) for m in range(6) for t in range(2)]
stat_col_names = [f"p{p+1}_m{m+1}_{stat}" for p in range(2) for m in range(6) for stat in stat_names]
adv_col_names = [f"p1_m{i+1}_adv_over_p2_m{j+1}" for i in range(6) for j in range(6)]

In [10]:
# add the total advantage column for ease later
full_match_data["p1_total_adv"] = full_match_data[adv_col_names].sum(axis=1)

# initialize the reduced stat column names (replacing atk and spa by off)
red_stat_col_names = copy.deepcopy(stat_col_names)
# Add the offensive stat column and modify red_stat_col_names
for player_index in range(2):
    for mon_index in range(6):
        full_match_data[f"p{player_index + 1}_m{mon_index+1}_off"] = full_match_data[[f"p{player_index + 1}_m{mon_index + 1}_atk",f"p{player_index+1}_m{mon_index+1}_spa"]].max(axis=1)
        red_stat_col_names.remove(f"p{player_index + 1}_m{mon_index + 1}_atk")
        red_stat_col_names.remove(f"p{player_index + 1}_m{mon_index + 1}_spa")
        red_stat_col_names.append(f"p{player_index + 1}_m{mon_index + 1}_off")

In [11]:
# These dataframes are for selecting out subsets of the data.
# We should throw away matches where people rage quit early
complete_matches = full_match_data[(full_match_data['duration'] > 60) & ((full_match_data["p1_revealed_team_size"] > 2) | (full_match_data["p2_revealed_team_size"] > 2))]
# This is just to easily grab matches where we know the players ratings
rated_matches = complete_matches[complete_matches['p1_rating'] > 0]
# This is to grab matches where we know that the players understand the basic switching strategy (from Marz' work on switching)
highly_rated_matches = rated_matches[rated_matches['p1_rating'] > 1300]

In [12]:
highly_rated_matches

,id,rated,duration,p1,p2,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,...,p1_m3_off,p1_m4_off,p1_m5_off,p1_m6_off,p2_m1_off,p2_m2_off,p2_m3_off,p2_m4_off,p2_m5_off,p2_m6_off
1,gen9randombattle-2631763570,False,167,PineappleCats,L4V,1959,10,False,6,6,...,213,186,203,267,205,317,253,254,216,234
2,gen9randombattle-2631369343,True,275,Chicken347,cococem,1999,-69,True,6,6,...,261,270,244,259,250,271,248,196,185,189
3,gen9randombattle-2631529004,False,123,WhatEver2102,Duck Cop,1999,17,True,1,3,...,180,288,250,243,239,236,176,239,232,145
4,gen9randombattle-2631993792,False,301,monomythic,OverthereStair,2120,58,False,6,6,...,190,280,257,258,218,204,234,152,201,247
5,gen9randombattle-2631629401,False,116,Elite 4 Waally,jsjsiis,1958,38,False,5,4,...,217,219,301,255,182,235,183,269,217,188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4780,gen9randombattle-2631800171,False,468,green8Bit,joebidenfan1245,2078,72,True,5,6,...,188,211,272,243,259,214,254,288,204,216
4781,gen9randombattle-2631440114,False,376,DiamondIyes,100taka,1953,41,True,6,6,...,217,247,247,240,185,205,155,182,141,232
4782,gen9randombattle-2631435014,False,180,itsclover,JhuniorCq,1973,-41,False,5,4,...,206,226,250,176,251,196,185,254,249,219
4783,gen9randombattle-2632021860,False,212,who is to say,cuerodeolancho,2304,119,True,6,6,...,211,219,208,200,238,211,221,166,164,223


In [13]:
full_match_data['p1_wins'].value_counts(normalize=True)

p1_wins
False    0.519223
True     0.480777
Name: proportion, dtype: float64

In [14]:
highly_rated_matches['p1_wins'].value_counts(normalize=True)

p1_wins
False    0.523073
True     0.476927
Name: proportion, dtype: float64

In [15]:
full_match_data['elo_diff'].mean()

np.float64(-12.459882992060175)

In [16]:
highly_rated_matches['elo_diff'].mean()

np.float64(-14.000245459008346)

In [17]:
# sns.violinplot(full_match_data[['elo_diff','p1_wins']],x='elo_diff',hue='p1_wins')


In [18]:
# sns.violinplot(highly_rated_matches[['elo_diff','p1_wins']],x = 'elo_diff',hue='p1_wins')

In [19]:
#!{sys.executable} -m pip install fg-data-profiling

In [20]:
# import data_profiling
# from data_profiling import ProfileReport

In [21]:
# profile = ProfileReport(highly_rated_matches[['p1_rating','elo_diff','p1_total_adv'] + adv_col_names + ['p1_wins']],title = 'EDA_of_adv_in_highly_rated')
# profile.to_html()
# profile.to_file("advantage_in_highly_rated_matches_report.html")

In [22]:
skf = StratifiedKFold(n_splits=10,shuffle=True,random_state=207)

log_reg_elo_diff = LogisticRegression(penalty=None,max_iter=1000,n_jobs=-1)

accs = []
lls = []
roc_aucs = []
for train_ind,test_ind in skf.split(highly_rated_matches[['elo_diff']],highly_rated_matches['p1_wins']):
    log_reg_elo_diff.fit(highly_rated_matches[['elo_diff']].iloc[train_ind],highly_rated_matches['p1_wins'].iloc[train_ind])
    preds = log_reg_elo_diff.predict(highly_rated_matches[['elo_diff']].iloc[test_ind])
    pred_probas = log_reg_elo_diff.predict_proba(highly_rated_matches[['elo_diff']].iloc[test_ind])
    accs.append(accuracy_score(highly_rated_matches['p1_wins'].iloc[test_ind],preds))
    lls.append(log_loss(highly_rated_matches['p1_wins'].iloc[test_ind],pred_probas))
    roc_aucs.append(roc_auc_score(highly_rated_matches[['p1_wins']].iloc[test_ind],pred_probas[:,1]))

print("Baseline model (logistic regression on Elo differential):")
print(f"\t Accuracy mean = {np.mean(accs)} with standard deviation = {np.std(accs,ddof=1)}")
print(f"\t Log-loss mean = {np.mean(lls)} with standard deviation = {np.std(lls,ddof=1)}")
print(f"\t ROC-AUC mean = {np.mean(roc_aucs)} with standard deviation = {np.std(roc_aucs,ddof=1)}")

Baseline model (logistic regression on Elo differential):
	 Accuracy mean = 0.5238130510189334 with standard deviation = 0.013184047226506524
	 Log-loss mean = 0.6916278668803478 with standard deviation = 0.0014729217029172124
	 ROC-AUC mean = 0.5207280556901845 with standard deviation = 0.02327438612718948


In [23]:
skf = StratifiedKFold(n_splits=10,shuffle=True,random_state=207)
log_reg = LogisticRegression(penalty=None,max_iter=1000,n_jobs=-1)
rand_forest = RandomForestClassifier(n_estimators=100,criterion='log_loss',n_jobs=-1)
rand_forest_with_type = Pipeline([
    ('one_hot', ColumnTransformer(
        [('types', OneHotEncoder(handle_unknown='ignore'), type_col_names)],
        remainder='passthrough'
    )),
    ('forest', RandomForestClassifier(n_estimators=100,criterion='log_loss',n_jobs=-1))
])

models = [
    ('log_reg_elo_diff',log_reg),
    ('log_reg_stats_elo_diff',log_reg),
    ('rand_forest_types_stats_elo_diff',rand_forest_with_type),
    ('log_reg_adv_elo_diff',log_reg),
    ('log_reg_adv_stats_elo_diff',log_reg),
    ('rand_forest_adv_elo_diff',rand_forest),
    ('rand_forest_adv_types_stats_elo_diff',rand_forest_with_type),
    ('log_reg_total_adv_elo_diff',log_reg)
]

target = highly_rated_matches['p1_wins']
feature_sets = [
    highly_rated_matches[['elo_diff']],
    highly_rated_matches[red_stat_col_names + ['elo_diff']],
    highly_rated_matches[type_col_names + stat_col_names + ['elo_diff']],
    highly_rated_matches[adv_col_names + ['elo_diff']],
    highly_rated_matches[adv_col_names + red_stat_col_names + ['elo_diff']],
    highly_rated_matches[adv_col_names + ['elo_diff']],
    highly_rated_matches[adv_col_names + type_col_names + stat_col_names + ['elo_diff']],
    highly_rated_matches[['p1_total_adv','elo_diff']]
]

scoring = {
    'accuracy' : 'accuracy',
    'log_loss' : 'neg_log_loss',
    'roc_auc' : 'roc_auc'
}

def summarize(name, cvres):
    return pd.Series({
        "model": name,
        "roc_auc_mean":  cvres["test_roc_auc"].mean(),
        "roc_auc_std":   cvres["test_roc_auc"].std(ddof=1),
        "log_loss_mean": -cvres["test_log_loss"].mean(),
        "log_loss_std":  (-cvres["test_log_loss"]).std(ddof=1),
        "acc_mean":      cvres["test_accuracy"].mean(),
        "acc_std":       cvres["test_accuracy"].std(ddof=1),
    })

rows = []
for ind in range(len(models)):
    model = models[ind][1]
    cv = cross_validate(model,feature_sets[ind],target,cv=skf,n_jobs=-1,scoring=scoring)
    rows.append(summarize(models[ind][0],cv))

summary = pd.DataFrame(rows)
summary

,model,roc_auc_mean,roc_auc_std,log_loss_mean,log_loss_std,acc_mean,acc_std
0,log_reg_elo_diff,0.520728,0.023274,0.691628,0.001473,0.523813,0.013184
1,log_reg_stats_elo_diff,0.517574,0.018855,0.699145,0.004542,0.518164,0.016526
2,rand_forest_types_stats_elo_diff,0.489815,0.036283,0.701140,0.006976,0.503670,0.035291
3,log_reg_adv_elo_diff,0.518803,0.021381,0.695582,0.004642,0.524799,0.018168
4,log_reg_adv_stats_elo_diff,0.520750,0.024090,0.703507,0.008686,0.521101,0.021104
5,rand_forest_adv_elo_diff,0.504783,0.021562,0.701373,0.005243,0.510059,0.019405
6,rand_forest_adv_types_stats_elo_diff,0.504382,0.027965,0.697737,0.005814,0.510549,0.021746
7,log_reg_total_adv_elo_diff,0.535495,0.037449,0.690778,0.004560,0.531912,0.020019


In [24]:
highly_rated_matches

,id,rated,duration,p1,p2,p1_rating,elo_diff,p1_wins,p1_revealed_team_size,p2_revealed_team_size,...,p1_m3_off,p1_m4_off,p1_m5_off,p1_m6_off,p2_m1_off,p2_m2_off,p2_m3_off,p2_m4_off,p2_m5_off,p2_m6_off
1,gen9randombattle-2631763570,False,167,PineappleCats,L4V,1959,10,False,6,6,...,213,186,203,267,205,317,253,254,216,234
2,gen9randombattle-2631369343,True,275,Chicken347,cococem,1999,-69,True,6,6,...,261,270,244,259,250,271,248,196,185,189
3,gen9randombattle-2631529004,False,123,WhatEver2102,Duck Cop,1999,17,True,1,3,...,180,288,250,243,239,236,176,239,232,145
4,gen9randombattle-2631993792,False,301,monomythic,OverthereStair,2120,58,False,6,6,...,190,280,257,258,218,204,234,152,201,247
5,gen9randombattle-2631629401,False,116,Elite 4 Waally,jsjsiis,1958,38,False,5,4,...,217,219,301,255,182,235,183,269,217,188
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4780,gen9randombattle-2631800171,False,468,green8Bit,joebidenfan1245,2078,72,True,5,6,...,188,211,272,243,259,214,254,288,204,216
4781,gen9randombattle-2631440114,False,376,DiamondIyes,100taka,1953,41,True,6,6,...,217,247,247,240,185,205,155,182,141,232
4782,gen9randombattle-2631435014,False,180,itsclover,JhuniorCq,1973,-41,False,5,4,...,206,226,250,176,251,196,185,254,249,219
4783,gen9randombattle-2632021860,False,212,who is to say,cuerodeolancho,2304,119,True,6,6,...,211,219,208,200,238,211,221,166,164,223
